In [9]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import utils, boto3
import pandas as pd
import sagemaker.core.image_uris as sm_image_uris
pd.set_option('display.max_colwidth', None)  # show full text in each cell
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap

[06/15/26 17:58:25] INFO     Found credentials from IAM      ]8;id=6712274;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=6712275;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\
                             Role: ec2-1                                        
sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [3]:
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
data_bucket='omm-test-bucket'
project_path = 'models/abalone'
account='088461143167'

role = utils.get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)

In [8]:
utils.view_model_groups(boto_session)

,ModelPackageGroupName,ModelPackageGroupArn,CreationTime,ModelPackageGroupStatus
0,abalone,arn:aws:sagemaker:us-east-1:088461143167:model-package-group/abalone,2026-06-15 17:56:51.124000+00:00,Completed


In [7]:
# make model group
sm_client.create_model_package_group(ModelPackageGroupName='abalone',ModelPackageGroupDescription='Abalone age regression models')

{'ModelPackageGroupArn': 'arn:aws:sagemaker:us-east-1:088461143167:model-package-group/abalone',
 'ResponseMetadata': {'RequestId': 'dd051aa8-74fe-4589-9cab-8a6827cdee33',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'dd051aa8-74fe-4589-9cab-8a6827cdee33',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '95',
   'date': 'Mon, 15 Jun 2026 17:56:51 GMT'},
  'RetryAttempts': 0}}

In [11]:
utils.view_model_versions(boto_session)

g: {'ModelPackageGroupName': 'abalone', 'ModelPackageGroupArn': 'arn:aws:sagemaker:us-east-1:088461143167:model-package-group/abalone', 'ModelPackageGroupDescription': 'Abalone age regression models', 'CreationTime': datetime.datetime(2026, 6, 15, 17, 56, 51, 124000, tzinfo=tzlocal()), 'ModelPackageGroupStatus': 'Completed'}
v: {'ModelPackageGroupName': 'abalone', 'ModelPackageVersion': 1, 'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/1', 'CreationTime': datetime.datetime(2026, 6, 15, 17, 58, 32, 74000, tzinfo=tzlocal()), 'ModelPackageStatus': 'Completed', 'ModelApprovalStatus': 'PendingManualApproval', 'ModelPackageRegistrationType': 'Registered'}


,ModelPackageGroupName,ModelPackageVersion,ModelPackageArn,ModelPackageDescription,CreationTime,ModelPackageStatus,ModelApprovalStatus,ModelPackageRegistrationType
0,abalone,1,arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/1,,2026-06-15 17:58:32.074000+00:00,Completed,PendingManualApproval,Registered


In [10]:
# register
sm_client.create_model_package(
    ModelPackageGroupName='abalone',  # use group name, not ModelPackageName
    InferenceSpecification={
        'Containers': [
            {
                'Image': sm_image_uris.retrieve('xgboost', region, version='1.5-1'),
                'ModelDataUrl': 's3://omm-test-bucket/abalone/models/abalone/model/abalone-train-job-20260614232059/output/model.tar.gz'
            }
        ],
        'SupportedContentTypes': ['text/csv'],
        'SupportedResponseMIMETypes': ['text/csv'],
        'SupportedRealtimeInferenceInstanceTypes': ['ml.m5.large', 'ml.m5.xlarge'],
        'SupportedTransformInstanceTypes': ['ml.m5.large', 'ml.m5.xlarge']
    },
    ModelApprovalStatus='PendingManualApproval'
)

[06/15/26 17:58:31] INFO     Ignoring unnecessary instance     ]8;id=6712282;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=6712283;file:///home/ec2-user/.local/lib/python3.9/site-packages/sagemaker/core/image_uris.py#535\535]8;;\
                             type: None.                                        


{'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/1',
 'ResponseMetadata': {'RequestId': '17f36f11-e3cb-46a8-b27e-b815625904e6',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '17f36f11-e3cb-46a8-b27e-b815625904e6',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '86',
   'date': 'Mon, 15 Jun 2026 17:58:32 GMT'},
  'RetryAttempts': 0}}